# 03b · Pachetul pentru etichetarea irigării
Pregătește în QGIS ce trebuie etichetat, ca să nu digitizezi nimic:

- `data/out/irrigation_labels.gpkg`: 300 de parcele cu cultură de primăvară, răspândite pe toată zona. Coloana `irigat` e goală și o completezi tu cu 1 (irigat) sau 0 (neirigat). Cel puțin 150–200 de parcele.
- `data/interim/s2_iulie_2024.tif` și `data/interim/s2_august_2024.tif`: imagini Sentinel-2 pentru fundal (roșu, verde, albastru, infraroșu apropiat), 10 m.

**Cum etichetezi:** parcela e irigată dacă e verde în plină secetă (iulie–august), în timp ce culturile de primăvară din jur sunt uscate. Pivoții circulari sunt exemple sigure. Dacă nu ești sigur, las-o goală.

Indiciul folosit la selecție rămâne separat, în `data/interim/irrigation_label_hints.parquet`, ca să nu-ți influențeze judecata. Notebook-ul nu suprascrie etichetele existente.

In [ ]:
import os, sys
if not os.path.exists('gisday_config.py') and os.path.exists('../gisday_config.py'):
    os.chdir('..')
sys.path.insert(0, os.getcwd())
from gisday_config import *
from gisday_lib import select_label_candidates
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
import pystac_client, planetary_computer
parcels = gpd.read_parquet(OUT / 'parcels.parquet')
F = pd.read_parquet(OUT / 'parcel_features.parquet')[['parcel_id', 'ndvi_ja_2024']]
T = pd.read_parquet(OUT / 'parcel_target.parquet')[['parcel_id', 'is_arable', 'crop_group_2024']]
D = parcels.merge(F, on='parcel_id').merge(T, on='parcel_id')
c = D.geometry.centroid
D['block'] = (c.x // 5000).astype(int).astype(str) + '_' + (c.y // 5000).astype(int).astype(str)

In [ ]:
LABELS = OUT / 'irrigation_labels.gpkg'
if LABELS.exists():
    print('Etichetele există deja; nu le suprascriu:', LABELS)
else:
    pick = select_label_candidates(D)
    L = D.merge(pick, on='parcel_id')
    L[['parcel_id', 'hint']].to_parquet(INTERIM / 'irrigation_label_hints.parquet')
    L['irigat'] = pd.array([pd.NA] * len(L), dtype='Int32')
    L[['parcel_id', 'irigat', 'area_ha', 'geometry']].to_file(LABELS, layer='etichete', driver='GPKG')
    print(len(L), 'parcele de etichetat;', L.hint.value_counts().to_dict())

In [ ]:
# Imaginile de fundal: scena cea mai curată din tile-ul 34TGP care acoperă toată zona
cat = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1', modifier=planetary_computer.sign_inplace)

def clearest(window):
    items = cat.search(collections=['sentinel-2-l2a'], bbox=AOI_BBOX, datetime=window,
                       query={'eo:cloud_cover': {'lt': 10}, 's2:mgrs_tile': {'eq': '34TGP'}}).item_collection()
    return min((i for i in items if i.properties.get('sat:relative_orbit') == 93), key=lambda i: i.properties['eo:cloud_cover'])

def write_scene(item, path):
    if path.exists():
        print('există:', path.name); return
    bands = ['B04', 'B03', 'B02', 'B08']
    with rasterio.open(item.assets['B04'].href) as ref:
        west, south, east, north = rasterio.warp.transform_bounds('EPSG:4326', ref.crs, AOI_BBOX[0] - .02, AOI_BBOX[1] - .02, AOI_BBOX[2] + .02, AOI_BBOX[3] + .02)
        win = from_bounds(west, south, east, north, ref.transform).round_offsets().round_lengths()
        profile = ref.profile | dict(driver='GTiff', count=4, width=int(win.width), height=int(win.height), transform=ref.window_transform(win),
                                     compress='deflate', predictor=2, tiled=True, blockxsize=512, blockysize=512, nodata=0)
    with rasterio.open(path, 'w', **profile) as dst:
        for i, b in enumerate(bands, 1):
            with rasterio.open(item.assets[b].href) as src:
                dst.write(src.read(1, window=win), i)
        dst.build_overviews([2, 4, 8, 16], Resampling.average)
    print('scris:', path.name, item.id, item.properties['eo:cloud_cover'])

import rasterio.warp
write_scene(clearest(f'{YEAR}-07-20/{YEAR}-07-31'), INTERIM / 's2_iulie_2024.tif')
write_scene(clearest(f'{YEAR}-08-05/{YEAR}-08-25'), INTERIM / 's2_august_2024.tif')